In [13]:
import climt
from sympl import DataArray, TendencyComponent, AdamsBashforth
from sympl import (
    PlotFunctionMonitor, NetCDFMonitor,
    TimeDifferencingWrapper, UpdateFrequencyWrapper,
    set_constant, get_constant, initialize_numpy_arrays_with_properties
)
import gfs_dynamical_core
from datetime import timedelta
import numpy as np
import torch
import torch.nn as nn
import sys
sys.path.append("..")
from models import DynamicMLP, DynamicMLP_flatten

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
class TorchScaler:
    def __init__(self, mean, std):
        self.mean = mean  # [C]
        self.std = std
    
    def transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return (x - mean) / std
    
    def inverse_transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return x * std + mean

class NNParameterization(TendencyComponent):

    input_properties = {
        'air_temperature': {'dims': ['*', 'mid_levels'], 'units': 'degK'},
        'specific_humidity': {'dims': ['*', 'mid_levels'], 'units': 'kg/kg'},
        'surface_air_pressure': {'dims': ['*'], 'units': 'Pa'},
        'surface_upward_latent_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
        'surface_upward_sensible_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
    }

    tendency_properties = {
        'air_temperature': {
            'units': 'degK day^-1'
        }
    }
    
    diagnostic_properties = {}

    def __init__(self, **kwargs):
        super(NNParameterization, self).__init__(**kwargs)

        self.IN_FEATURES = 59
        self.OUT_FEATURES = 28

        ckpt = torch.load("./best_model/best_model_trial_0.pth",map_location=device)
        ckpt_norm = torch.load('/projects/sds-lab/Shuochen/climt/gmd_aquaplanet/64x32_normalization.pth', map_location=device)
        self.input_scaler  = TorchScaler(ckpt_norm['X_mean'], ckpt_norm['X_std'])
        self.output_scaler = TorchScaler(ckpt_norm['y_mean'], ckpt_norm['y_std'])

        self.model = DynamicMLP_flatten(self.IN_FEATURES,self.OUT_FEATURES,ckpt["hidden_sizes"]).to(device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def array_call(self, state):

        num_cols, num_levs = state['air_temperature'].shape
        
        tendencies = initialize_numpy_arrays_with_properties(
            self.tendency_properties, state, self.input_properties
        )
        diagnostics = initialize_numpy_arrays_with_properties(
            self.diagnostic_properties, state, self.input_properties
        )
        # ---------------------------------
        # Extract state
        # ---------------------------------
        T = state['air_temperature']        # ()
        q = state['specific_humidity']      # ()
        ps = state['surface_air_pressure']  # ()
        lh = state['surface_upward_latent_heat_flux'] # ()
        sh = state['surface_upward_sensible_heat_flux'] # ()
        # ---------------------------
        # Build NN input []
        # ---------------------------        
        x = np.concatenate([
            T,
            q,
            ps[..., None],
            lh[..., None],
            sh[..., None],
        ], axis=-1)  # ()
        
        # normalize
        if self.input_scaler is not None:
            x = self.input_scaler.transform(x)
            
        # Add batch dimension
        x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
        # ---------------------------------
        # NN inference
        # ---------------------------------
        with torch.no_grad():
            y = self.model(x)   # ()
        y = y.cpu().numpy().squeeze()
        # y = y / 86400.0
        # print(y.shape)
        # inverse normalize if needed
        if self.output_scaler is not None:
            y = self.output_scaler.inverse_transform(y)
            
        # ---------------------------------
        # Map NN output → temperature tendency
        # ---------------------------------
        # da = DataArray(y, dims=['*', 'mid_levels'])
        # # da.attrs['units'] = 'degK day^-1'
        
        # da.attrs['units'] = 'degK s^-1'
        # tendencies = {'air_temperature_tendency_from_longwave': da}
        tendencies['air_temperature'][:] = y[:, :num_levs] #shape -> lon*lat, cols

        return tendencies, diagnostics
        
model_time_step = timedelta(minutes=10)
# Create components
convection = climt.EmanuelConvection(tendencies_in_diagnostics=True)
simple_physics = TimeDifferencingWrapper(climt.SimplePhysics())
radiation_step = timedelta(hours=1)
# radiation_lw = UpdateFrequencyWrapper(climt.RRTMGLongwave(), radiation_step)
radiation_sw = UpdateFrequencyWrapper(climt.RRTMGShortwave(), radiation_step)
slab_surface = climt.SlabSurface()
# nn component
nn_component = NNParameterization()

dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, nn_component, convection], number_of_damped_levels=5)
# dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, radiation_lw, convection], number_of_damped_levels=5)

grid = climt.get_grid(nx=64, ny=32)
my_state = climt.get_default_state([dycore], grid_state=grid)

timestep = timedelta(minutes=10)
n_steps = 14400  # 1 day at 10-min timestep
for i in range(n_steps):
    # print(my_state.keys())
    diag, my_state = dycore(my_state, model_time_step)
    my_state.update(diag)
    # print(my_state['air_temperature'].attrs)
    # ?
    # diag['air_temperature_tendency_from_longwave'].attrs['units'] = 'degK'
    # diagnostics, my_state = time_stepper(my_state, model_time_step)
    # print(my_state.keys())
    
    if i % 10 == 0:
        T_mid = my_state['air_temperature'].values[15].mean()
        print(f"Step {i}, mean T_mid = {T_mid:.2f} K")

Step 0, mean T_mid = 290.00 K
Step 10, mean T_mid = 290.03 K
Step 20, mean T_mid = 290.06 K
Step 30, mean T_mid = 290.09 K
Step 40, mean T_mid = 290.11 K
Step 50, mean T_mid = 290.14 K
Step 60, mean T_mid = 290.17 K
Step 70, mean T_mid = 290.19 K
Step 80, mean T_mid = 290.22 K
Step 90, mean T_mid = 290.24 K
Step 100, mean T_mid = 290.26 K
Step 110, mean T_mid = 290.29 K
Step 120, mean T_mid = 290.31 K
Step 130, mean T_mid = 290.33 K
Step 140, mean T_mid = 290.35 K
Step 150, mean T_mid = 290.38 K
Step 160, mean T_mid = 290.40 K
Step 170, mean T_mid = 290.42 K
Step 180, mean T_mid = 290.44 K
Step 190, mean T_mid = 290.46 K
Step 200, mean T_mid = 290.48 K
Step 210, mean T_mid = 290.50 K
Step 220, mean T_mid = 290.52 K
Step 230, mean T_mid = 290.54 K
Step 240, mean T_mid = 290.55 K
Step 250, mean T_mid = 290.57 K
Step 260, mean T_mid = 290.59 K
Step 270, mean T_mid = 290.61 K
Step 280, mean T_mid = 290.62 K
Step 290, mean T_mid = 290.64 K
Step 300, mean T_mid = 290.66 K
Step 310, mean T_mi

/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/sympl/_core/base_components.py:522: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if properties['units'] is '':
/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/sympl/_core/base_components.py:522: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if properties['units'] is '':


KeyboardInterrupt: 